In [ ]:
import os
import re
import json
import logging
import sys
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# (tùy chọn khi debug CUDA) bật dòng sau trước khi import torch ở môi trường dev:
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# =============== Logging hiển thị trong console/Jupyter ===============
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)

# =============== Cấu hình đầu vào/đầu ra ===============
INPUT_XLSX  = "Confessions of HNMU.xlsx"  # sửa nếu cần
OUTPUT_JSON = "output_toxic_2.json"         # file json dạng mảng
COL_NAME    = "post_text"                 # tên cột văn bản trong Excel

# =============== Tải mô hình ViHateT5 ===============
custom_cache_dir = "D:/Model"             # cache vào ổ D
os.makedirs(custom_cache_dir, exist_ok=True)

MODEL_NAME = "tarudesu/ViHateT5-base-HSD"

logging.info("⏳ Đang load ViHateT5...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, cache_dir=custom_cache_dir, local_files_only=False
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, cache_dir=custom_cache_dir, local_files_only=False
)

# Dẹp cảnh báo pad_token_id
if getattr(model.config, "pad_token_id", None) is None:
    model.config.pad_token_id = model.config.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()
logging.info(f"✅ ViHateT5 đã sẵn sàng trên: {device}")

# Nhãn 3 lớp phổ biến; gom OFFENSIVE/HATE => toxic
BIN_TOXIC = {"OFFENSIVE", "HATE"}

def normalize_label(text: str) -> str:
    t = (text or "").strip().upper()
    if "OFFEN" in t: return "OFFENSIVE"
    if "HATE"  in t: return "HATE"
    if "CLEAN" in t or "NORMAL" in t: return "CLEAN"
    # nếu lạ → bảo thủ coi là CLEAN
    return "CLEAN"

# Thay thế toàn bộ hàm classify_vihatet5_label bằng phiên bản "scoring":
@torch.inference_mode()
def classify_vihatet5_label(text: str, on_device: str) -> str:
    # 1) Encode văn bản
    enc = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    if on_device != "cpu":
        enc = {k: v.to(on_device) for k, v in enc.items()}

    # 2) Tập nhãn ứng viên (HSD 3 lớp)
    candidates = ["CLEAN", "OFFENSIVE", "HATE"]

    # 3) Tính -loss (≈ log-likelihood) cho từng nhãn bằng teacher-forcing
    scores = {}
    for cand in candidates:
        lab_ids = tokenizer(cand, return_tensors="pt", add_special_tokens=True).input_ids
        if on_device != "cpu":
            lab_ids = lab_ids.to(on_device)

        # model(**enc, labels=lab_ids) trả về loss: càng thấp càng tốt
        out = model(**enc, labels=lab_ids)
        scores[cand] = -out.loss.item()  # đảo dấu để "cao hơn tốt hơn"

    # 4) Chọn nhãn có score cao nhất
    best = max(scores, key=scores.get)
    logging.info(f"Scores {text[:24]}... -> {scores}")
    return best


def classify_post(text: str) -> dict:
    """Trả về dict: method, status, label (3-class). Không dùng regex."""
    try:
        label = classify_vihatet5_label(text, device)
    except RuntimeError as e:
        # Fallback: nếu GPU có sự cố bất chợt, chạy post này trên CPU
        logging.error(f"GPU error: {e} → fallback CPU cho post này.")
        model.to("cpu")
        label = classify_vihatet5_label(text, "cpu")
        model.to(device)  # đưa về device ban đầu cho post tiếp theo

    status = "toxic" if label in BIN_TOXIC else "non-toxic"
    return {
        "method": "vihatet5",
        "status": status,
        "toxic_word": None,   # không dùng regex nên để null
        "label": label        # giữ nhãn 3-class để bạn debug
    }

if __name__ == "__main__":
    # 1) Đọc Excel
    df = pd.read_excel(INPUT_XLSX)
    if COL_NAME not in df.columns:
        raise ValueError(f"Không tìm thấy cột '{COL_NAME}' trong file Excel.")
    texts = df[COL_NAME].astype(str).tolist()

    # 2) Ghi JSON dạng mảng, từng post một
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        f.write("[\n")
        for idx, text in enumerate(texts):
            logging.info(f"🔎 Đang xử lý post {idx+1}/{len(texts)}; nội dung: {text[:80]}...")
            try:
                pred = classify_post(text)
                result = {
                    "index": idx,
                    "post_text": text,
                    **pred
                }
            except Exception as e:
                logging.error(f"[{idx}] Lỗi classify: {e}")
                result = {
                    "index": idx,
                    "post_text": text,
                    "method": "vihatet5",
                    "status": "error",
                    "toxic_word": None,
                    "label": "ERROR",
                    "error": str(e)
                }

            line = json.dumps(result, ensure_ascii=False, indent=4)
            if idx > 0: f.write(",\n")
            f.write(line)
            f.flush()
        f.write("\n]\n")

    logging.info(f"✅ Kết quả đã được lưu vào: {OUTPUT_JSON}")


2025-10-02 19:27:32,338 - INFO - ⏳ Đang load ViHateT5...
2025-10-02 19:27:35,447 - INFO - ✅ ViHateT5 đã sẵn sàng trên: cuda
2025-10-02 19:27:35,536 - INFO - 🔎 Đang xử lý post 1/1015; nội dung: Địt...
2025-10-02 19:27:35,596 - INFO - Scores Địt... -> {'CLEAN': -6.277884483337402, 'OFFENSIVE': -2.5453484058380127, 'HATE': -4.097559452056885}
2025-10-02 19:27:35,602 - INFO - 🔎 Đang xử lý post 2/1015; nội dung: 8149: Mình được các cô ở phòng CTQLHSSV nhắc là báo với các bạn dù đã, sắp và ch...
2025-10-02 19:27:35,705 - INFO - Scores 8149: Mình được các cô ở... -> {'CLEAN': -0.2026747316122055, 'OFFENSIVE': -2.194228410720825, 'HATE': -2.432422399520874}
2025-10-02 19:27:35,705 - INFO - 🔎 Đang xử lý post 3/1015; nội dung: 8147: Mn ơi cho e hỏi bao giờ trường xét vb2 ạ 8148: bao h có ds hb dự kiến v ạ ...
2025-10-02 19:27:35,781 - INFO - Scores 8147: Mn ơi cho e hỏi ba... -> {'CLEAN': -0.23265402019023895, 'OFFENSIVE': -1.6322579383850098, 'HATE': -1.875051736831665}
2025-10-02 19:27:35,782 